Attention Mechanism has memeory and computation complexity of O(n^2), n is sequential length. It is computationally impossible to fed the entire dataset into this hardware. 

With 72 classes and distribution ranging from ~400k to ~2.5k samples, we need to address this bias



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import random

MAX_SEQ_LENGTH = 512
SAMPLES_PER_CLASS = 50
INPUT_CSV_PATH = "C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/02_processed/comprehensive_master_data_universal.parquet"
OUTPUT_CSV_PATH = "GripFormer_preprocessed_dataset.csv"

In [ ]:
df = pd.read_parquet(INPUT_CSV_PATH)

print(f'Initial dataframe shape: {df.shape}')

In [ ]:
# Drop duplicate rows
df.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {df.shape}")

## 2. Feature Engineering and Selection

In [ ]:
df["trial_id"] = df["subjName"].astype(str) + "_" + df["trialN"].astype(str)

feature_cols = [
    'indexX_unified', 'indexY_unified', 'indexZ_unified',
    'thumbX_unified', 'thumbY_unified', 'thumbZ_unified',
    'wristX_unified', 'wristY_unified', 'wristZ_unified',
    'FX', 'FY', 'FZ', 'FVel', 'FAcc', 'MVel', 'MAcc', 'MDec',
    'signal_grasp'
]

# Convert boolean
df["signal_grasp"] = df["signal_grasp"].astype(int)

# Keep only the necessary columns
df_processed = df[["trial_id", "grip_strategy_label"] + feature_cols]
print(f"Selected {len(feature_cols)} features for the model")

## 3. Grouping into sequences

In [ ]:
# Get the label for each trial (they are constant within a trial)
trial_labels = df_processed.groupby("trial_id")["grip_strategy_label"].first()

# Group feature by trial
grouped_features = df_processed.groupby("trial_id")[feature_cols].apply(lambda x: x.to_numpy())

# Combine into a single structure
sequences = pd.concat([grouped_features, trial_labels], axis=1)
sequences.columns = ["features", "label"]
print(f"Created {len(sequences)} unique sequences")

## 4. Handling Class Imbalance

In [ ]:
print(f"Balancing dataset to {SAMPLES_PER_CLASS} samples per class...")
class_distribution_before = sequences["label"].value_counts()
print("Class distribution before balancing: \n", class_distribution_before)

balanced_sequences = []
for label, group in sequences.groupby("label"):
    class_sequences = group.to_dict("records")

    if len(class_sequences) > SAMPLES_PER_CLASS:
        balanced_sequences.extend(random.sample(class_sequences, SAMPLES_PER_CLASS))
    else:
        balanced_sequences.extend(random.choices(class_sequences, k=SAMPLES_PER_CLASS))

random.shuffle(balanced_sequences)
print(f"Total sequences after balancing: {len(balanced_sequences)}")

## 5. Padding and Trucating Sequences

In [ ]:
print(f"Padding/Truncating all sequences to MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}...")
final_data = []
for i, seq_data in enumerate(tqdm(balanced_sequences, desc="Processing sequences")):
    features = seq_data["features"]
    label = seq_data["label"]
    seq_len = features.shape[0]

    if seq_len > MAX_SEQ_LENGTH:
        processed_features = features[:MAX_SEQ_LENGTH, :]
    else:
        padding_height = MAX_SEQ_LENGTH - seq_len
        padding = np.zeros((padding_height, features.shape[1]))
        processed_features = np.vstack([features, padding])
    
    for timestep in range(MAX_SEQ_LENGTH):
        row = {
            'unique_sequence_id': i, 
            'timestep': timestep,
            'label': label,
        }
        for j, col_name in enumerate(feature_cols):
            row[col_name] = processed_features[timestep, j]
        final_data.append(row)

## Final creation

In [ ]:
final_df = pd.DataFrame(final_data)

label_encoder = LabelEncoder()
final_df["label_encoded"] = label_encoder.fit_transform(final_df["label"])

label_mapping = {index: label for index, label in enumerate(label_encoder.classes_)}
print("\n--- Label Encoding Mapping ---")
for index, label in label_mapping.items():
    print(f"{index}: {label}")
print("-----------------\n")
# Reorder columns for clarity
final_cols = ['unique_sequence_id', 'timestep', 'label', 'label_encoded'] + feature_cols
final_df = final_df[final_cols]

In [ ]:
print("Final dataset preview:")
print(final_df.head())
print(f"\nFinal dataset shape: {final_df.shape}")
print(f"Number of unique sequences in final dataset: {final_df['unique_sequence_id'].nunique()}")


In [ ]:
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nSuccess! Preprocessed dataset saved to '{OUTPUT_CSV_PATH}'")
